In [1]:
# Install hypertools (run this first on Colab)
import importlib.util
if importlib.util.find_spec('hypertools') is None:
    %pip install -q "hypertools[interactive,lsl]"

# Streaming from a Lab Streaming Layer (LSL) device

[Lab Streaming Layer](https://labstreaminglayer.org) (LSL) is the de facto
standard for streaming time-synchronized data -- EEG, eye tracking,
motion capture, physiological signals, and more -- from acquisition
hardware/software into downstream analysis tools over the local network.
`hyp.io.lsl_stream()` (GH #130) resolves a live LSL stream and wraps it as
a plain Python iterator of per-sample numeric vectors -- exactly the shape
`hyp.plot`'s streaming support (GH #101, see the
[streaming data tutorial](streaming_data.ipynb)) already expects from any
generator -- so an LSL stream plugs straight into `hyp.plot(...,
stream_init=..., stream_chunk=...)` with no extra glue.

`pylsl` (which wraps the native `liblsl` library used by essentially every
LSL-speaking device/app) is HyperTools' `[lsl]` extra. `hyp.io.lsl_stream()`
installs it on demand the first time it is called; to fetch it ahead of time,
run `pip install "hypertools[lsl]"`.

This tutorial doesn't require any real hardware: it starts a **synthetic**
LSL outlet on a background thread (publishing multi-channel oscillating
data), so the whole example runs anywhere, on any machine, with no
external device attached -- but `hyp.io.lsl_stream()` itself is exactly
the same function you'd use with a real EEG amp, eye tracker, or any other
LSL-speaking device.


In [1]:


import hypertools as hyp

%matplotlib inline


## A synthetic LSL outlet

`hyp.io.lsl.synthetic_outlet` publishes a 6-channel signal -- each channel a sine wave at a slightly different frequency, plus noise -- through a real `pylsl.StreamOutlet` running on a background thread. It is the same outlet hypertools' own test suite uses to exercise `hyp.io.lsl_stream()` without hardware; stop it with `outlet.stop()` (or use it as a context manager).


In [2]:
N_CHANNELS = 6
STREAM_NAME = 'HypertoolsTutorialStream'

outlet = hyp.io.lsl.synthetic_outlet(STREAM_NAME, n_channels=N_CHANNELS)
print(f'synthetic LSL outlet {STREAM_NAME!r} running on a background thread')


synthetic LSL outlet 'HypertoolsTutorialStream' running on a background thread


2026-09-05 03:17:33.929 (   0.001s) [         637E551]         api_config.cpp:126   INFO| Loaded default config
2026-09-05 03:17:33.929 (   0.001s) [         637E551]             common.cpp:78    INFO| git:64988c6a14b8dc3b3f270ece58eab4f480bfab43/branch:refs/tags/v1.17.7/build:Release/compiler:AppleClang-17.0.0.17000013/link:SHARED


## Resolving the stream and plotting it live

`hyp.io.lsl_stream(name=..., timeout=5.0)` resolves the outlet above by its
LSL `name` property. `timeout` is how long to wait for a matching outlet to
appear on the network before giving up (the same value also bounds
mid-stream silence once samples are flowing). The call returns a plain
iterator of per-sample channel vectors, which is exactly what `hyp.plot`'s
streaming path consumes.

`hyp.plot` treats it exactly like the synthetic generator in the
[streaming data tutorial](streaming_data.ipynb): `stream_init=200` primes
the plot with the first 200 samples (and fits the reduction model on them),
then `stream_chunk=20` samples are pulled and appended per animation frame.
`stream_max=600` stops after 600 samples have been pulled from the device.
Everything consumed stays on the figure; pass `stream_window` if you want
older samples to scroll off.

In [3]:
stream = hyp.io.lsl_stream(name=STREAM_NAME, timeout=5.0)

fig = hyp.plot(stream, stream_init=200, stream_chunk=20, stream_max=600,
               title='Live LSL stream (synthetic outlet)',
               save_path='lsl_streaming.mp4', frame_rate=5, show=False)
fig.stream_info['n_samples'], fig.stream_info['xform_data'][0].shape

(600, (600, 3))

<video controls loop muted autoplay playsinline src="lsl_streaming.mp4" title="A live LSL stream, plotted as it arrives" style="max-width: 100%"></video>

[Download the clip](lsl_streaming.mp4)

## Cleaning up the synthetic outlet

Stop the background-thread outlet once we're done streaming from it (a
real device's outlet is managed by its own acquisition software, so this
step is specific to this tutorial's synthetic stand-in).

`hyp.io.lsl_stream` returns an `LSLStream`: `stream.close()` (or leaving a
`with stream:` block) destroys the underlying LSL inlet, and hypertools also
releases it when the stream is garbage-collected or the interpreter exits, so
a stream you simply stop using never logs liblsl's alarming (but harmless)
'stream transmission broke off' error at teardown. We still close it *first*
here, before stopping the outlet thread, so the shutdown is orderly.


In [4]:
stream.close()  # close our LSL inlet before tearing down the outlet
outlet.stop()
print('outlet thread stopped:', outlet.closed)


outlet thread stopped: True


## Using a real device

Everything above works identically with a real LSL-speaking device or
application (an EEG amplifier, eye tracker, motion capture rig, etc.) --
just resolve it by `type=` instead of `name=` (or both):

```python
# resolve any live EEG-type stream on the local network
stream = hyp.io.lsl_stream(type='EEG', timeout=10.0)
hyp.plot(stream, stream_init=200, stream_chunk=20)
```

See [labstreaminglayer.org](https://labstreaminglayer.org) for the full
list of LSL-speaking acquisition software, and the
[`hyp.io.lsl_stream` API reference](../api.rst) for the full parameter
list.
